In [1]:
import os
import sys
import pickle
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase

sys.path.insert(0, str(Path("..").resolve()))
load_dotenv("../.env")

True

In [2]:
from graph.embedder import Embedder
from graph.kin_rag import QueryAwareRetriever, CypherNativeRetriever, AnswerAgent, StatelessCypherRetriever

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

DATASET_NAME = "musique"  # "musique" or "2wiki"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
embedder = Embedder(cache_path=Path("cache/embeddings_cache.pkl"))

QUERY_ENTITY_CACHE_PATH = Path(f"cache/query_entities-{DATASET_NAME}.pkl")
if QUERY_ENTITY_CACHE_PATH.exists():
    with open(QUERY_ENTITY_CACHE_PATH, "rb") as f:
        query_entity_cache = pickle.load(f)
    print(f"[QueryEntityCache] Loaded {len(query_entity_cache)} cached query entities from {QUERY_ENTITY_CACHE_PATH}")
else:
    query_entity_cache = {}
    print(f"[QueryEntityCache] No cache found at {QUERY_ENTITY_CACHE_PATH}, starting empty")


def save_query_entity_cache() -> None:
    QUERY_ENTITY_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(QUERY_ENTITY_CACHE_PATH, "wb") as f:
        pickle.dump(query_entity_cache, f)

[Embedder] Loaded 828 cached embeddings
[QueryEntityCache] Loaded 999 cached query entities from cache/query_entities-musique.pkl


In [3]:
retriever_class = StatelessCypherRetriever  # CypherNativeRetriever  # QueryAwareRetriever

# Toggle to compare query-aware vs uniform propagation on the full 1000-sample run.
# Set False to run the no-query-aware (uniform) baseline; results save to a separate CSV.
QUERY_AWARE = False
RUN_TAG = "" if QUERY_AWARE else "no-query-aware"

retriever = retriever_class(
    driver=driver,
    embedder=embedder,
    max_steps=3,
    decay=0.7,
    resource_threshold=0.01,
    seed_vector_fallback_k=5,
    top_k_entities=30,
    top_k_paths=30,
    query_aware=QUERY_AWARE,
    query_entity_cache=query_entity_cache,
    verbose=True,
)

agent = AnswerAgent(retriever=retriever, llm_judge=True)
print(f"Retriever: {retriever_class.__name__}  |  propagation: {RUN_TAG}")

Retriever: StatelessCypherRetriever  |  propagation: no-query-aware


In [4]:
with open(f"ground_truth-{DATASET_NAME}.pkl", "rb") as f:
    ground_truth = pickle.load(f)

GT_INDEX = 5
print(f"Loaded {len(ground_truth)} questions")
ground_truth[GT_INDEX]

Loaded 1000 questions


{'question_id': 'q_5',
 'question': "Who is the child of Caroline LeRoy's spouse?",
 'answer': 'Fletcher Webster',
 'supporting_paragraphs': ['Fletcher Webster__b8719790aa',
  'Caroline LeRoy__c7a8eebd86']}

## Single question demo

In [5]:
gt = ground_truth[GT_INDEX]
print(f"Q: {gt['question']}")
print(f"A: {gt['answer']}")
print(f"Supporting: {gt['supporting_paragraphs']}\n")

result = agent.evaluate(gt["question"], gt["answer"])

print(f"\nPrediction: {result['prediction']}")
print(f"EM: {result['em']}, F1: {result['f1']:.3f}")
print(f"Seeds: {result['retrieval'].seed_entities}")
print(f"Extracted: {result['retrieval'].extracted_entities}")
print(f"Steps: {result['retrieval'].steps_taken}")
print(f"Paths: {len(result['retrieval'].paths)}")

Q: Who is the child of Caroline LeRoy's spouse?
A: Fletcher Webster
Supporting: ['Fletcher Webster__b8719790aa', 'Caroline LeRoy__c7a8eebd86']


Prediction: caroline leroy webster
EM: 0.0, F1: 0.400
Seeds: ['leroy', 'herman leroy', 'le roy', 'caroline leroy webster', 'cæsar boeck']
Extracted: [QueryEntity(name='caroline leroy', search_hint='caroline leroy person')]
Steps: 3
Paths: 18


In [ ]:
print("--- Knowledge graph paths (multi-hop chains) ---")
for p in result["retrieval"].paths:
    names, edges = p["names"], p["edges"]
    chain = names[0]
    for i, edge in enumerate(edges):
        rel = edge["relation"]
        if edge["forward"]:
            chain += f" --[{rel}]--> {names[i+1]}"
        else:
            chain += f" <--[{rel}]-- {names[i+1]}"
    print(f"  w={p['path_weight']:.4f}  {chain}")

print("\n--- Entity descriptions ---")
for e in result["retrieval"].entities:
    desc = e["description"]
    print(f"  {e['resource']:.4f}  {e['name']:30s}  {desc}")

print("\n--- Context sent to LLM ---")
print(result["paths_context"])
print(result["descriptions_context"])

## Batch evaluation

In [6]:
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed

from tqdm import tqdm

N_EVAL = 1000
REPORT_EVERY = 25
WINDOW = 100
MAX_WORKERS = 5

assert isinstance(retriever, StatelessCypherRetriever), (
    "Parallel evaluation requires StatelessCypherRetriever. "
    "Set retriever_class = StatelessCypherRetriever in the setup cell."
)

eval_results = [None] * N_EVAL
recent_em = deque(maxlen=WINDOW)
recent_f1 = deque(maxlen=WINDOW)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    future_to_idx = {
        pool.submit(agent.evaluate, gt["question"], gt["answer"]): i
        for i, gt in enumerate(ground_truth[:N_EVAL])
    }

    progress = tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="Evaluating")
    for done_count, fut in enumerate(progress, start=1):
        idx = future_to_idx[fut]
        gt = ground_truth[idx]
        try:
            result = fut.result()
            eval_results[idx] = result
            recent_em.append(float(result["em"]))
            recent_f1.append(float(result["f1"]))
        except Exception as e:
            print(f"Error on {gt['question_id']}: {e}")

        if done_count % REPORT_EVERY == 0:
            valid = [r for r in eval_results if r is not None]
            n = max(1, len(valid))
            progress.set_postfix({
                "EM": f"{sum(r['em'] for r in valid) / n:.3f}",
                "F1": f"{sum(r['f1'] for r in valid) / n:.3f}",
                f"EM@{WINDOW}": f"{sum(recent_em) / max(1, len(recent_em)):.3f}",
                f"F1@{WINDOW}": f"{sum(recent_f1) / max(1, len(recent_f1)):.3f}",
                "ok": len(valid),
            })
            save_query_entity_cache()

save_query_entity_cache()
ok = sum(1 for r in eval_results if r is not None)
print(f"\n[QueryEntityCache] Saved {len(query_entity_cache)} cached query entities to {QUERY_ENTITY_CACHE_PATH}")
print(f"Completed {ok}/{N_EVAL} questions successfully")

Evaluating:   0%|          | 0/1000 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 1000/1000 [13:45<00:00,  1.21it/s, EM=0.261, F1=0.343, EM@100=0.260, F1@100=0.329, ok=1000]


[QueryEntityCache] Saved 999 cached query entities to cache/query_entities-musique.pkl
Completed 1000/1000 questions successfully


In [7]:
import pandas as pd

rows = [
    AnswerAgent.flatten_result(result, ground_truth[i]["question_id"])
    for i, result in enumerate(eval_results)
    if result is not None
]
df = pd.DataFrame(rows)
df.to_csv(f"eval-results-{DATASET_NAME}-{RUN_TAG}.csv", index=False)

# Print aggregates
print(f"Results on {len(df)} questions [{RUN_TAG}]:")
print(f"  EM:  {df['em'].mean():.3f}")
print(f"  F1:  {df['f1'].mean():.3f}")
print(f"  LLM Accuracy:  {df['llm_accuracy'].mean():.3f}")
print(f"  Avg prompt tokens: {df['prompt_tokens'].mean():.0f}")
print(f"  Avg completion tokens: {df['completion_tokens'].mean():.0f}")
print(f"  Avg paths: {df['num_paths'].mean():.1f}")
print(f"  Avg entities: {df['num_entities'].mean():.1f}")

Results on 1000 questions [no-query-aware]:
  EM:  0.261
  F1:  0.343
  LLM Accuracy:  0.301
  Avg prompt tokens: 17537
  Avg completion tokens: 7
  Avg paths: 16.2
  Avg entities: 28.3


## Query-aware vs no-query-aware comparison (n=1000)

Compares all four full-dataset runs: `{musique, 2wiki} × {query-aware, no-query-aware}`, reading the per-run `eval-results-{dataset}-{mode}.csv` files.

To produce them, run the batch above four times, flipping `DATASET_NAME` (setup cell) and `QUERY_AWARE` (config cell). The cell below reads whatever is present and skips missing runs.

In [10]:
import pandas as pd

DATASETS = ["musique", "2wiki"]
MODES = ["query-aware", "no-query-aware"]

comparison_rows = []
for dataset in DATASETS:
    for mode in MODES:
        path = Path(f"eval-results-{dataset}-{mode}.csv")
        if not path.exists():
            print(f"[skip] {path} not found")
            continue
        cdf = pd.read_csv(path)
        comparison_rows.append({
            "dataset": dataset,
            "config": mode,
            "n": len(cdf),
            "EM": round(cdf["em"].mean(), 4),
            "F1": round(cdf["f1"].mean(), 4),
            "LLM-acc": round(cdf["llm_accuracy"].mean(), 4),
            "prompt_tok": round(cdf["prompt_tokens"].mean()),
            "paths": round(cdf["num_paths"].mean(), 1),
            "entities": round(cdf["num_entities"].mean(), 1),
        })

comparison_df = pd.DataFrame(comparison_rows)

# Query-aware advantage per dataset (query-aware − no-query-aware)
for dataset in DATASETS:
    sub = comparison_df[comparison_df["dataset"] == dataset]
    qa = sub[sub["config"] == "query-aware"]
    no_qa = sub[sub["config"] == "no-query-aware"]
    if len(qa) and len(no_qa):
        qa, no_qa = qa.iloc[0], no_qa.iloc[0]
        print(
            f"[{dataset}] ΔF1 = {qa['F1'] - no_qa['F1']:+.4f}   "
            f"ΔEM = {qa['EM'] - no_qa['EM']:+.4f}   "
            f"ΔLLM-acc = {qa['LLM-acc'] - no_qa['LLM-acc']:+.4f}   "
            f"(query-aware − no-query-aware)"
        )

comparison_df

[musique] ΔF1 = +0.0741   ΔEM = +0.0670   ΔLLM-acc = +0.0790   (query-aware − no-query-aware)
[2wiki] ΔF1 = +0.0355   ΔEM = +0.0350   ΔLLM-acc = +0.0350   (query-aware − no-query-aware)


,dataset,config,n,EM,F1,LLM-acc,prompt_tok,paths,entities
0,musique,query-aware,1000,0.328,0.4168,0.380,11838,15.0,27.0
1,musique,no-query-aware,1000,0.261,0.3427,0.301,17537,16.2,28.3
2,2wiki,query-aware,1000,0.478,0.5499,0.540,4532,18.3,28.2
3,2wiki,no-query-aware,1000,0.443,0.5144,0.505,5249,19.6,28.4


In [ ]:
driver.close()